In [1]:
!pip install gdown

In [2]:
import gdown

In [3]:
file_id = "1-SWJ_nIgotQ11ZHapb-uqWndvzeRs80d"
output_file = "Chest_XRay_Datasets.zip"

# Download the file
gdown.download(f"https://drive.google.com/uc?id={file_id}", output_file, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1-SWJ_nIgotQ11ZHapb-uqWndvzeRs80d
From (redirected): https://drive.google.com/uc?id=1-SWJ_nIgotQ11ZHapb-uqWndvzeRs80d&confirm=t&uuid=e6a833c6-a67e-4203-b3cc-498ef244e732
To: /content/Chest_XRay_Datasets.zip
100%|██████████| 1.01G/1.01G [01:02<00:00, 16.1MB/s]


'Chest_XRay_Datasets.zip'

In [4]:
import zipfile
z = zipfile.ZipFile('/content/Chest_XRay_Datasets.zip')
z.extractall()

In [5]:
import os
import shutil

src = 'Chest_XRay_Datasets'
dst_dir = 'datasets'
dst = os.path.join(dst_dir, src)

os.makedirs(dst_dir, exist_ok=True)

if os.path.exists(dst):
  shutil.rmtree(dst)

shutil.move(src, dst)

print(f"moved'{src}' to {dst} succesfuly.")


moved'Chest_XRay_Datasets' to datasets/Chest_XRay_Datasets succesfuly.


In [6]:
import yaml

data = {

        'path':'Chest_XRay_Datasets',
        'train':'train',
        'test':'test',
        'val':'train',
         'nc' : 2,
         'names': ['Normal', 'Pnemonia']

}



#saving to pothole.yaml
with open('Chest_XRay_Datasets.yaml', 'w') as outfile:
    yaml.dump(data, outfile, default_flow_style=False)


print("Chest_XRay_Datasets.yaml created successfully")

Chest_XRay_Datasets.yaml created successfully


In [7]:
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Load a YOLOv11 classification model
model = YOLO('yolo11l-cls.pt')  # Replace with 'yolo11m-cls.pt' or similar for better accuracy

# Train the classification model
model.train(
    data='Chest_XRay_Datasets',
    epochs=5,
    imgsz=224,
    batch=8,
    name='Chest_XRay_Datasets',
    save=True,
    save_period=-1,
    patience=20,
    val=True,
    degrees=15,       # Rotation
    flipud=0.3,       # Vertical flip probability
    fliplr=0.5,       # Horizontal flip probability
    scale=0.5,        # Image scaling
    shear=10,         # Shear angle
    translate=0.1     # Translation
)



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Chest_XRay_Datasets, degrees=15, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import os
import random

# Load trained classification model
model = YOLO('runs/classify/Chest_XRay_Datasets/weights/best.pt')

# Test images directory
test_images_folders = 'datasets/Chest_XRay_Datasets/test'

# Get list of image files
image_folders = os.listdir(test_images_folders)

# Plot settings
fig, ax = plt.subplots(4, 4, figsize=(16, 16))
ax = ax.ravel()

for idx in range(16):
    img_folder = random.choice(image_folders)
    img_files = os.listdir(os.path.join(test_images_folders,img_folder))
    img_path = os.path.join(test_images_folders, img_folder, random.choice(img_files))
    image = cv2.imread(img_path)

    # Perform classification inference
    results = model(img_path)  # returns a list with one result
    result = results[0]

    # Get predicted class name
    class_id = int(result.probs.top1)
    class_name = model.names[class_id]
    confidence = result.probs.top1conf.item()

    # Convert BGR to RGB for plotting
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Plot the image with class name and confidence
    ax[idx].imshow(image_rgb)
    ax[idx].set_title(f"Actual: {img_folder}\n Predicted: {class_name} ({confidence:.2f})", fontsize=12)
    ax[idx].axis('off')

plt.tight_layout()
plt.show()